# 10b — Containers, configuration, and deployment — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/10.md) · [Course map](../PLAN.md) · [Project](../../projects/stage10/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Explain image/container/network/volume boundaries
- Validate runtime configuration
- Design release and rollback evidence


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Containers, configuration, and restore drills

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

An image packages an application and dependencies. A container is a running instance. A volume stores data outside a disposable container's writable layer. A network lets services address each other; localhost inside one container refers to that container, not another service.

Configuration describes deployment-specific choices. Secrets are credentials and belong in protected runtime configuration rather than source control. A reverse proxy can handle public HTTPS while the API and database communicate privately.

A backup is a data copy; a restore test proves the copy can become usable data. A release identifies source, dependencies, image, schema, and model/index versions. Rollback switches to a known-good compatible version; changing the app image does not automatically reverse a database migration.

### Worked example: follow the values

Create a record, restart only the API, and read it again. Then back up the database, restore into a separate database, compare counts and exact sums, and read it through the application repository. Protect the original database while practicing recovery.

### Pause and explain

Why is a backup file on disk insufficient evidence of recovery?

<details><summary>Check your reasoning after trying</summary>

It may be incomplete, incompatible, or unusable. A separate restore plus data/application checks exercises the recovery path.

</details>

### Common mistakes to check

A floating image tag is not an immutable release identity. An in-memory rate limiter does not coordinate multiple API workers.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

An image is a filesystem and runtime specification; a container is a running instance. A volume retains data beyond a container’s lifecycle. A network connects services by service name; localhost inside a container refers to that container. Bind only intended interfaces and ports. A reverse proxy terminates HTTPS/TLS and routes requests; DNS resolves names.

Keep configuration outside the image. Secrets belong in a secret store or protected runtime environment, never source control or image layers. Use a non-root user, a small build context, dependency locking, and scanned/pinned release artifacts. Development version ranges are not a release lock.

CI should run tests, evaluation gates, and image builds. Deployment should apply tested migrations, check readiness, and support rollback. Rolling back an image cannot undo an incompatible database migration. In Coolify or another platform, configure persistent volumes, private networks, TLS, health checks, and backups deliberately. Local Docker success is one checkpoint, not evidence of operational readiness.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
config = {"host": "database", "port": 5432}
print("Containers connect using service DNS:", config)


## Exercise 1: Required configuration

Read a mapping with DATABASE_URL and API_KEY; reject missing/blank values or API_KEY shorter than 24 characters.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def validate_config(env):
    result = {k: env.get(k, "").strip() for k in ("DATABASE_URL", "API_KEY")}
    if not all(result.values()) or len(result["API_KEY"]) < 24: raise ValueError("Incomplete configuration")
    return result


In [ ]:
assert validate_config({"DATABASE_URL": "postgresql://db/example", "API_KEY": "x"*32})["API_KEY"] == "x"*32
expect_error(ValueError, lambda: validate_config({}))
print("Exercise 1: checks passed")


**Why this works:** The length check is only a local training guard, not a complete credential policy.


## Exercise 2: Release gate

Return True only when tests, evals, migration_rehearsal, and restore_drill are all exactly True.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def release_ready(evidence):
    return all(evidence.get(k) is True for k in ("tests", "evals", "migration_rehearsal", "restore_drill"))


In [ ]:
assert not release_ready({"tests": True, "evals": True})
assert release_ready(dict.fromkeys(["tests", "evals", "migration_rehearsal", "restore_drill"], True))
print("Exercise 2: checks passed")


**Why this works:** A release checklist is useful only when each item points to an actual artifact.


## Exercise 3: Schema compatibility

A service supports schema versions inclusive [minimum,maximum]. Return whether current is compatible; reject inverted bounds.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def compatible(current, minimum, maximum):
    if minimum > maximum: raise ValueError("Invalid range")
    return minimum <= current <= maximum


In [ ]:
assert compatible(3, 2, 4)
assert not compatible(5, 2, 4)
print("Exercise 3: checks passed")


**Why this works:** Expand-contract migrations can keep old and new application versions compatible during rollout.


## Independent transfer

Use the provided Dockerfile/Compose, run API+PostgreSQL, test persistence after restart, and rehearse backup/restore into a separate database. Add CI and document a Coolify deployment and rollback without publishing secrets.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [10b interface and acceptance cases](../assignments/10b.md).
2. Copy the [blank starter](../assignments/starters/10b.py) to `work/assignments/10b.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 10b --solution work/assignments/10b.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/10b.md#10b-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 10b --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why is localhost often wrong in Compose?

   It points at the current container rather than the database service.

2. Why pin release artifacts?

   To know exactly which dependencies and image are deployed and make rollback reproducible.


## Reading and review

- [Primary reference 1](https://docs.docker.com/get-started/)
- [Primary reference 2](https://docs.docker.com/compose/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
